In [1]:
!pip install rasterio numpy 
# install the libraries
# don't need the ! outside of jupyter

In [2]:
import rasterio
import numpy as np
import os
#import libraries

In [3]:
raster_path = r"C:\Users\svbai\OneDrive\Desktop\flood_mask_2324_BNG.tif" # path to flood mask raster, projected in British National Grid

In [4]:
flood_value = 1 # value representing flooded pixels

In [5]:
with rasterio.open(raster_path) as src:
    raster = src.read(1) # read the first band
    nodata = src.nodata # get NoData value if there is one
    
    pixel_width = src.transform.a # get pixel size
    pixel_height = abs(src.transform.e)
    pixel_area = pixel_width * pixel_height

    if nodata is not None: # create a valid data mask
        valid_mask = raster != nodata
    else:
        valid_mask = np.ones(raster.shape, dtype=bool)

    flood_pixels = np.sum((raster == flood_value) & valid_mask) # count flooded pixels
    total_valid_pixels = np.sum(valid_mask) # count valid pixels
    flooded_area_m2 = flood_pixels * pixel_area # calculate flooded area
    flooded_area_km2 = flooded_area_m2 / 1_000_000

# print results
print(f"Raster: {os.path.basename(raster_path)}")
print(f"Flooded pixels: {flood_pixels}")
print(f"Total valid pixels: {total_valid_pixels}")
print(f"Pixel area: {pixel_area:.2f} m2")
print(f"Flooded area: {flooded_area_m2:.2f} m2")
print(f"Flooded area: {flooded_area_km2:.2f} km2")
print("CRS:", src.crs)
print(f"Unique raster values:", np.unique(raster))
print("Pixel width", pixel_width)
print("Pixel height", pixel_height)

Raster: flood_mask_2324_BNG.tif
Flooded pixels: 51877
Total valid pixels: 1966152
Pixel area: 701.03 m2
Flooded area: 36367074.25 m2
Flooded area: 36.37 km2
CRS: EPSG:27700
Unique raster values: [  0   1 255]
Pixel width 26.476876823968077
Pixel height 26.476876823968098


In [6]:
# calculate percentage area flooded

percent_flooded = (flood_pixels / total_valid_pixels) * 100
print(f"Percent flooded: {percent_flooded:.2f}%")

Percent flooded: 2.64%


## Export Results to a CSV

In [8]:
import pandas as pd

os.makedirs(r"C:\Users\svbai\B01051807_Bain_EGM722_Programming_Project\outputs", exist_ok=True) # create outputs folder if it doesn't already exist

results = {
    "raster": os.path.basename(raster_path),
    "flood_pixels": flood_pixels,
    "total_valid_pixels": total_valid_pixels,
    "pixel_area_m2": pixel_area,
    "flooded_area_m2": flooded_area_m2,
    "flooded_area_km2": flooded_area_km2,
    "percent_flooded": percent_flooded
}

df = pd.DataFrame([results])

df.to_csv(r"C:\Users\svbai\B01051807_Bain_EGM722_Programming_Project\outputs\flood_summary.csv", index=False)
print(df)
print("CSV saved to outputs/flood_summary.csv")

                    raster  flood_pixels  total_valid_pixels  pixel_area_m2  \
0  flood_mask_2324_BNG.tif         51877             1966152     701.025006   

   flooded_area_m2  flooded_area_km2  percent_flooded  
0     3.636707e+07         36.367074         2.638504  
CSV saved to outputs/flood_summary.csv
